# GeoSR-4 — DINOv2 / DINOv3-sat493m perceptual-loss backbone ablation (Colab GPU)

D023's perceptual loss uses VGG16, ImageNet-pretrained on natural photos -- a real domain mismatch for Sentinel-2 satellite imagery. Research (D042) found DINOv3 has a checkpoint pretrained on SAT-493M (satellite imagery) that's a much better domain match -- gated access has since been **approved**, so section 4 below uses the real satellite-pretrained backbone.

This notebook has two runs:
- **Section 3**: `facebook/dinov2-small` (free, no gating) -- isolates whether a stronger backbone helps on its own architectural merits, independent of domain match.
- **Section 4**: `facebook/dinov3-vitl16-pretrain-sat493m` (satellite-pretrained, gated -- access approved) -- the real test of whether domain-matched pretraining helps.

Both match D028/D029's VGG "quality run" protocol as closely as possible (30 epochs, lr 1e-4, lambda-perceptual 0.01, ICNR on) -- only `--perceptual-backbone`/`--dino-model-id` and, for section 4, `--batch-size` (smaller model needs smaller batch, see section 4's note) change, so eval numbers stay comparable to the already-logged D028/D029 baseline.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
`transformers` is new here -- needed to load the DINOv2 backbone.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision transformers

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Train: DINOv2 perceptual backbone (same protocol as D028/D029's VGG quality run)
Only `--perceptual-backbone dino` changed vs the original quality run -- epochs, batch size, lr, lambda-perceptual, ICNR (default on) all identical, so the eval numbers below are directly comparable to D028/D029.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # D025: fragmentation fix

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 8 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --amp \
  --checkpoint-dir experiments/swinir_dino_quality \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 4. DINOv3 sat493m (satellite-pretrained) run
Gated access **approved** (D042) -- this now uses the actual satellite-domain backbone, the real point of this whole detour. `facebook/dinov3-vitl16-pretrain-sat493m` is ViT-L (300M params) vs DINOv2-small's 21M, so `--batch-size` is dropped to 4 as a conservative starting guess against T4's 15GB (untested -- reduce further if it OOMs, see D024/D025 for the same kind of VGG-perceptual-loss OOM fix pattern).

The login cell below opens an interactive prompt -- paste your HF token there (never in a code cell/chat), it only needs read access and to have accepted the DINOv3 license.

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted -- must have access to facebook/dinov3-vitl16-pretrain-sat493m

!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 4 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
  --amp \
  --checkpoint-dir experiments/swinir_dino3_sat_quality \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino3_sat_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Download the checkpoint

In [ ]:
from google.colab import files
files.download('experiments/swinir_dino_quality/swinir_epoch29.pt')
# uncomment once the DINOv3 sat493m run (section 4) has also completed:
# files.download('experiments/swinir_dino3_sat_quality/swinir_epoch29.pt')